# B2.11 · Building the SAST harness

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

Builds on **[B2.9 · Idempotency, replay and rollback](https://spbreed.github.io/cyber-commons/lessons/B2.9.html)**.

| | |
|---|---|
| Open-source tooling | OpenGrep, Trivy |
| Open-weight models | Kimi K2.6, GLM-5.2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The SAST harness is the first of four disciplines, and they all share the same
loop: **index → summarise → hypothesise → verify.** What makes each discipline
different is only the last step — what counts as proof.

For static analysis the shape is:

1. **Index.** Turn the repository into something navigable — symbols, call
   edges, entry points. Done once, reused by every later stage.
2. **Summarise.** Compress each component to something that fits in a context
   window, because the repository never will.
3. **Hypothesise.** Ask the model where a weakness could be, given the summary
   and the neighbourhood. This is the step people think is the whole system.
4. **Verify.** Check the hypothesis against the code — is the sink real, is the
   path reachable, is a sanitiser already present.

Two design decisions decide whether the output is usable:

**Per-file context budget.** More context stops helping and starts hurting past
a threshold that is specific to your corpus. A budget makes that measurable
rather than a matter of taste.

**Reachability gating before deduplication.** Gate first, then dedupe. Doing it
the other way round spends the expensive step on findings that were never
reachable.

The deduplication stage is what decides whether you ship *a queue* or *a
landfill*.

## 2 · The four stages, on a seeded corpus\n\nThe corpus has planted defects, so recall and precision are measurable rather than asserted.

In [ ]:
SEEDED = {
 # unit            (has_defect, cwe,      reachable_from_entry, sanitised)
 "get_report":     (True,  "CWE-22",  True,  False),
 "run_export":     (True,  "CWE-78",  True,  False),
 "legacy_import":  (True,  "CWE-89",  False, False),   # real bug, unreachable
 "render_row":     (False, None,      True,  False),
 "safe_query":     (True,  "CWE-89",  True,  True),    # real sink, sanitised
 "admin_purge":    (True,  "CWE-78",  True,  False),
 "format_date":    (False, None,      True,  False),
 "parse_config":   (False, None,      False, False),
}
planted = sorted(u for u, v in SEEDED.items() if v[0])
print(f"corpus units    : {len(SEEDED)}")
print(f"planted defects : {len(planted)}  {planted}")

## 3 · Index → summarise → hypothesise

In [ ]:
def index(corpus):
    return sorted(corpus)                                  # stage 1

def summarise(unit, budget_tokens):
    """Stage 2. A summary that fits the budget; detail is dropped, not hidden."""
    has_defect, cwe, reachable, sanitised = SEEDED[unit]
    full = {"unit": unit, "sink": bool(cwe), "cwe": cwe,
            "reachable": reachable, "sanitised": sanitised}
    if budget_tokens < 40:            # too small to carry the sanitiser fact
        full.pop("sanitised")
    if budget_tokens < 25:            # too small to carry reachability
        full.pop("reachable")
    return full

def hypothesise(summary):
    """Stage 3. Flags anything that looks like a sink. Deliberately noisy."""
    return summary["sink"]

for budget in (20, 30, 60):
    hyps = [u for u in index(SEEDED) if hypothesise(summarise(u, budget))]
    print(f"context budget {budget:>3} tokens -> {len(hyps)} hypotheses: {hyps}")

## 4 · Verify, and gate on reachability before you deduplicate

In [ ]:
def verify(unit, summary):
    """Stage 4. The cheap facts first; each one can only remove a finding."""
    if summary.get("sanitised"):
        return None, "a sanitiser is already present"
    if summary.get("reachable") is False:
        return None, "no path from any entry point"
    if summary.get("reachable") is None:
        return unit, "reachability unknown - budget too small to carry it"
    return unit, "reachable, unsanitised sink"

def pipeline(budget):
    hyps = [u for u in index(SEEDED) if hypothesise(summarise(u, budget))]
    kept, dropped = [], []
    for u in hyps:
        keep, why = verify(u, summarise(u, budget))
        (kept if keep else dropped).append((u, why))
    return hyps, kept, dropped

for budget in (20, 30, 60):
    hyps, kept, dropped = pipeline(budget)
    tp = [u for u, _ in kept if SEEDED[u][0]]
    fp = [u for u, _ in kept if not SEEDED[u][0]]
    recall = len(tp) / len(planted)
    prec = len(tp) / len(kept) if kept else 0
    print(f"budget {budget:>3}: {len(hyps)} hyp -> {len(kept)} kept  "
          f"recall {recall:.0%}  precision {prec:.0%}  fp={fp}")

## 5 · Where it breaks — more context is not monotonically better\n\nThe budget that carries every fact is not the budget with the best output, and the reason is worth understanding.

In [ ]:
print(f"{'budget':>7}{'kept':>6}{'recall':>9}{'precision':>11}  note")
for budget in (20, 25, 30, 40, 60):
    _, kept, _ = pipeline(budget)
    tp = [u for u, _ in kept if SEEDED[u][0]]
    prec = len(tp) / len(kept) if kept else 0
    note = ("reachability unknown - everything survives" if budget < 25 else
            "sanitiser fact missing - safe_query survives" if budget < 40 else
            "all facts present")
    print(f"{budget:>7}{len(kept):>6}{len(tp)/len(planted):>8.0%}{prec:>10.0%}  {note}")
print()
print("At budget 20 the harness keeps everything and calls it recall. It is not")
print("finding more - it is verifying less, and reporting the difference as a")
print("result. A recall number without its precision is a marketing number.")

## 6 · Verify — queue or landfill\n\nThe deduplication stage, and the number that decides which one you shipped.

In [ ]:
ANALYSERS = ("grep", "taint", "model")

_, kept, _ = pipeline(60)
raw = [(u, tool) for u, _ in kept for tool in ANALYSERS]     # each tool reports each
by_defect = sorted({u for u, _ in raw})

print(f"raw findings across {len(ANALYSERS)} analysers : {len(raw)}")
print(f"distinct defects                     : {len(by_defect)}")
print(f"inflation if you do not deduplicate  : {len(raw)/len(by_defect):.1f}x")
print()
minutes_per = 6
print(f"analyst minutes, undeduplicated : {len(raw)*minutes_per}")
print(f"analyst minutes, deduplicated   : {len(by_defect)*minutes_per}")
print()
print("Same findings, same accuracy, three times the queue. Nobody reads the")
print("third page, so the defect that gets fixed is whichever sorted first.")
assert len(raw) == 3 * len(by_defect)

## What you just proved

The four-stage loop runs over a seeded corpus of eight units with five planted defects. Shrinking the per-file context budget removes the facts verification depends on: below 25 tokens reachability is unknown and everything survives, which reads as higher recall and is actually less verification. Deduplicating three analysers' output collapses a 3x inflated queue back to the real defect count.

## Your turn

Find the context budget where your own corpus stops improving. It exists, it is specific to your codebase, and until you have measured it you are paying for tokens that make the output worse.

---

**Next → [B2.12 · Building the DAST and exploitation harness](https://spbreed.github.io/cyber-commons/lessons/B2.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*